## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, substring, col, concat, regexp_replace, upper, lpad, lower, when 

## Reading from bronze layer

In [0]:
df = spark.table("olist.bronze.sellers")
df.display()

## Overview about the table

In [0]:
print("=== Schema ===")
df.printSchema()

print("=== Row Count ===")
print(f"Total rows: {df.count()}")

print("=== Null Counts per Column ===")
df.select([
    F.count(F.when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

## Transformations

### 1. TRIM whitespace from all string columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


### 2. Normalize improperly represented nulls in string columns

In [0]:
NULL_STRINGS = ["", "null", "none", "n/a", "na", "unknown", "-", " "]

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            when(lower(trim(col(field.name))).isin(NULL_STRINGS), None)
            .otherwise(col(field.name))
        )

### 3. Standardize seller_city - Title Case & normalize spaces

In [0]:
df = df.withColumn(
    "seller_city",
    trim(regexp_replace(col("seller_city"), r"\s+", " "))  # normalize spaces first
)
df = df.withColumn(
    "seller_city",
    concat(
        upper(substring(col("seller_city"), 1, 1)),   # capitalize first letter
        lower(substring(col("seller_city"), 2, 9999)) # lowercase everything else
    )
)


### 4. Fix seller_zip_code_prefix - pad with leading zeros to 5 digits

In [0]:
df = df.withColumn(
    "seller_zip_code_prefix",
    lpad(col("seller_zip_code_prefix").cast("string"), 5, "0")
)

### 5. Standardize seller_state - UPPERCASE

In [0]:
df = df.withColumn(
    "seller_state",
    upper(trim(col("seller_state")))
)

### 6. Handle nulls - filter out rows missing critical keys

In [0]:
df = df.filter(col("seller_id").isNotNull())


### 7. Remove duplicates from seller_id

In [0]:
df = df.dropDuplicates(["seller_id"])

## Quality Checks 

In [0]:
print(f"Total rows after cleaning: {df.count()}")
print(f"Unique sellers: {df.select('seller_id').distinct().count()}")
print(f"Unique states: {df.select('seller_state').distinct().count()}")
print(f"Null seller_city: {df.filter(col('seller_city').isNull()).count()}")

df.display()

## Write to silver layer 

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("olist.silver.sellers")